# 12a — Validation Evaluation

This notebook answers the Week 6 validation requirements directly.

### Tasks
- Load the tuned model definitions from Week 5 CSV outputs.
- Recreate and fit the tuned models on the training data.
- Evaluate the models on the held-out validation data.
- Compare validation performance.
- Generate confusion matrices.
- Select the best model(s).

### Required outputs
- `outputs/metrics/validation_results.csv`
- validation confusion matrices
- validation comparison table
- validation comparison figure
- best-model summary and justification

> Week 5 saved model **summaries/parameters as CSV files**, not serialized `.pkl` or `.joblib` estimators.  
> Therefore, this notebook recreates the tuned estimators using the saved Week 5 parameters before validation.

## 1. Setup

In [ ]:
from pathlib import Path
import ast
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the project repository.")

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for folder in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)

## 2. Load the Week 5 tuning outputs

In [ ]:
week5_files = {
    "candidate_models": METRICS_DIR / "candidate_models.csv",
    "tuned_models": METRICS_DIR / "tuned_models.csv",
    "hyperparameter_tuning_summary": METRICS_DIR / "hyperparameter_tuning_summary.csv",
}

week5_tables = {}

for name, path in week5_files.items():
    if path.exists():
        week5_tables[name] = pd.read_csv(path)
        print(f"✓ Loaded {path.name}: {len(week5_tables[name])} row(s)")
    else:
        print(f"— Not found: {path.name}")

if not week5_tables:
    raise FileNotFoundError(
        "No Week 5 tuning CSV files were found in outputs/metrics."
    )

for name, table in week5_tables.items():
    print(f"\n{name}.csv")
    display(table)

## 3. Build the tuned-model table

The notebook first uses `tuned_models.csv`. If that file does not contain enough information, it falls back to `candidate_models.csv` and `hyperparameter_tuning_summary.csv`.

Column names are detected automatically so the notebook does not depend on one exact CSV layout.

In [ ]:
def find_col(df, choices):
    lower_map = {str(c).lower(): c for c in df.columns}
    for choice in choices:
        if choice.lower() in lower_map:
            return lower_map[choice.lower()]
    return None

def parse_params(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}

    if isinstance(value, dict):
        return value

    text = str(value).strip()
    if not text:
        return {}

    for parser in (ast.literal_eval, json.loads):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    return {}

# Prefer tuned_models.csv, then candidate_models.csv, then tuning summary.
source_name = None
source_df = None

for preferred in [
    "tuned_models",
    "candidate_models",
    "hyperparameter_tuning_summary",
]:
    if preferred in week5_tables and not week5_tables[preferred].empty:
        source_name = preferred
        source_df = week5_tables[preferred].copy()
        break

model_col = find_col(
    source_df,
    ["model", "model_name", "classifier", "estimator", "candidate_model"]
)

dataset_col = find_col(
    source_df,
    ["dataset", "dataset_name", "modality", "feature_set"]
)

params_col = find_col(
    source_df,
    [
        "best_params",
        "best_parameters",
        "parameters",
        "params",
        "hyperparameters",
        "tuned_parameters",
    ]
)

if model_col is None:
    raise ValueError(
        f"Could not identify a model-name column in {source_name}.csv. "
        f"Columns found: {list(source_df.columns)}"
    )

tuned_models = pd.DataFrame({
    "dataset": (
        source_df[dataset_col].astype(str)
        if dataset_col
        else "multimodal_full"
    ),
    "model": source_df[model_col].astype(str),
    "parameters": (
        source_df[params_col].apply(parse_params)
        if params_col
        else [{} for _ in range(len(source_df))]
    ),
})

tuned_models = tuned_models.drop_duplicates(
    subset=["dataset", "model"]
).reset_index(drop=True)

print(f"Using model definitions from: {source_name}.csv")
display(tuned_models)

## 4. Find the training and validation datasets

In [ ]:
# Optional manual overrides.
# Only fill these in if automatic discovery does not find the correct files.
MANUAL_DATA_FILES = {
    # Example:
    # "multimodal_full": {
    #     "train": DATA_DIR / "processed" / "multimodal_full_train.csv",
    #     "validation": DATA_DIR / "processed" / "multimodal_full_validation.csv",
    # }
}

all_csvs = []

for root in [DATA_DIR, OUTPUTS_DIR]:
    if root.exists():
        all_csvs.extend(root.rglob("*.csv"))

def clean_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def choose_file(dataset, kind):
    # Manual override wins.
    if dataset in MANUAL_DATA_FILES:
        value = MANUAL_DATA_FILES[dataset].get(kind)
        if value:
            p = Path(value)
            return p if p.exists() else None

    ds = clean_text(dataset)

    kind_words = {
        "train": ["train", "training"],
        "validation": ["validation", "valid", "val"],
    }[kind]

    candidates = []

    for p in all_csvs:
        name = clean_text(p.stem)

        # Ignore Week 5/Week 6 metric outputs.
        if any(
            bad in name
            for bad in [
                "candidate_models",
                "tuned_models",
                "hyperparameter_tuning",
                "validation_results",
                "confusion_matrix",
                "performance_summary",
                "metrics",
            ]
        ):
            continue

        if not any(word in name for word in kind_words):
            continue

        score = 0

        if ds and ds in name:
            score += 100

        ds_tokens = set(ds.split("_"))
        name_tokens = set(name.split("_"))
        score += len(ds_tokens & name_tokens)

        candidates.append((score, p))

    candidates.sort(key=lambda x: x[0], reverse=True)

    if candidates:
        return candidates[0][1]

    return None

data_records = []

for dataset in tuned_models["dataset"].unique():
    data_records.append({
        "dataset": dataset,
        "train_file": choose_file(dataset, "train"),
        "validation_file": choose_file(dataset, "validation"),
    })

data_inventory = pd.DataFrame(data_records)

display(data_inventory)

missing = data_inventory[
    data_inventory["train_file"].isna()
    | data_inventory["validation_file"].isna()
]

if not missing.empty:
    print("\nAutomatic discovery could not find every required train/validation file.")
    print("Available CSV files that contain 'train' or 'valid':")

    for p in all_csvs:
        n = p.name.lower()
        if "train" in n or "valid" in n:
            print(" -", p.relative_to(PROJECT_ROOT))

    print(
        "\nIf the correct files are listed above, add their paths to "
        "MANUAL_DATA_FILES and rerun this section."
    )

## 5. Recreate the tuned models

In [ ]:
def strip_pipeline_prefixes(params):
    cleaned = {}

    for key, value in params.items():
        new_key = str(key)

        for prefix in [
            "classifier__",
            "model__",
            "estimator__",
            "clf__",
        ]:
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]

        cleaned[new_key] = value

    return cleaned

def make_classifier(model_name, params):
    name = clean_text(model_name)
    params = strip_pipeline_prefixes(params)

    # Feature-selection parameters belong to the pipeline, not classifier.
    params = {
        k: v for k, v in params.items()
        if k not in {
            "k",
            "n_selected_features",
            "select__k",
            "selector__k",
            "feature_selection__k",
        }
    }

    if "dummy" in name:
        return DummyClassifier(**params)

    if "logistic" in name:
        params.setdefault("max_iter", 5000)
        return LogisticRegression(**params)

    if "random_forest" in name or "randomforest" in name:
        params.setdefault("random_state", 42)
        return RandomForestClassifier(**params)

    if "extra_trees" in name or "extratrees" in name:
        params.setdefault("random_state", 42)
        return ExtraTreesClassifier(**params)

    if "decision_tree" in name or name in {"tree", "dt"}:
        params.setdefault("random_state", 42)
        return DecisionTreeClassifier(**params)

    if "hist_gradient" in name or "histgradient" in name:
        params.setdefault("random_state", 42)
        return HistGradientBoostingClassifier(**params)

    if "gradient_boost" in name or "gradientboost" in name:
        params.setdefault("random_state", 42)
        return GradientBoostingClassifier(**params)

    if name in {"svc", "svm", "support_vector_machine"} or "support_vector" in name:
        return SVC(**params)

    if "knn" in name or "nearest_neighbor" in name:
        return KNeighborsClassifier(**params)

    raise ValueError(
        f"Model '{model_name}' is not yet mapped in make_classifier()."
    )

def get_selected_k(params):
    for key in [
        "select__k",
        "selector__k",
        "feature_selection__k",
        "n_selected_features",
        "k",
    ]:
        if key in params:
            try:
                return int(params[key])
            except Exception:
                pass
    return None

print("Model recreation function ready.")

## 6. Fit on training data and evaluate on validation

Preprocessing is fitted using the **training dataset only** and then applied to validation.  
This prevents validation-data leakage.

In [ ]:
results = []
prediction_store = {}

for _, model_row in tuned_models.iterrows():
    dataset = model_row["dataset"]
    model_name = model_row["model"]
    params = model_row["parameters"]

    data_row = data_inventory[data_inventory["dataset"] == dataset]

    if data_row.empty:
        print(f"✗ {dataset} | {model_name}: no dataset mapping")
        continue

    train_file = data_row.iloc[0]["train_file"]
    validation_file = data_row.iloc[0]["validation_file"]

    if pd.isna(train_file) or pd.isna(validation_file):
        print(f"✗ {dataset} | {model_name}: train/validation file missing")
        continue

    try:
        train_df = pd.read_csv(train_file, dtype={ID_COLUMN: str})
        validation_df = pd.read_csv(
            validation_file,
            dtype={ID_COLUMN: str},
        )

        if TARGET not in train_df.columns:
            raise ValueError(f"{TARGET!r} missing from training data")

        if TARGET not in validation_df.columns:
            raise ValueError(f"{TARGET!r} missing from validation data")

        feature_cols = [
            c for c in train_df.columns
            if c not in {
                ID_COLUMN,
                TARGET,
                "condition_group",
                "condition_original",
            }
        ]

        # Keep only columns also present in validation.
        feature_cols = [
            c for c in feature_cols
            if c in validation_df.columns
        ]

        X_train = train_df[feature_cols].copy()
        y_train = train_df[TARGET].copy()

        X_val = validation_df[feature_cols].copy()
        y_val = validation_df[TARGET].copy()

        numeric_cols = X_train.select_dtypes(
            include=["number", "bool"]
        ).columns.tolist()

        categorical_cols = [
            c for c in feature_cols
            if c not in numeric_cols
        ]

        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ])

        transformers = []

        if numeric_cols:
            transformers.append(
                ("num", numeric_pipeline, numeric_cols)
            )

        if categorical_cols:
            transformers.append(
                ("cat", categorical_pipeline, categorical_cols)
            )

        preprocessor = ColumnTransformer(
            transformers=transformers,
            remainder="drop",
        )

        classifier = make_classifier(model_name, params)
        selected_k = get_selected_k(params)

        steps = [("preprocess", preprocessor)]

        if selected_k is not None:
            steps.append(
                (
                    "feature_selection",
                    SelectKBest(
                        score_func=f_classif,
                        k=selected_k,
                    ),
                )
            )

        steps.append(("classifier", classifier))

        pipeline = Pipeline(steps)

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

        key = (
            f"{clean_text(dataset)}__"
            f"{clean_text(model_name)}"
        )

        prediction_store[key] = (y_val, y_pred)

        results.append({
            "dataset": dataset,
            "model": model_name,
            "n_validation": len(y_val),
            "accuracy": accuracy_score(y_val, y_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_val,
                y_pred,
            ),
            "macro_f1": f1_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "precision_macro": precision_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "recall_macro": recall_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "result_key": key,
        })

        print(f"✓ {dataset} | {model_name}")

    except Exception as e:
        print(f"✗ {dataset} | {model_name}: {e}")

validation_results = pd.DataFrame(results)

if validation_results.empty:
    raise RuntimeError(
        "No model completed validation. Review the messages above."
    )

validation_results = (
    validation_results
    .sort_values(
        ["macro_f1", "balanced_accuracy"],
        ascending=False,
    )
    .reset_index(drop=True)
)

validation_results.insert(
    0,
    "rank",
    range(1, len(validation_results) + 1),
)

display(validation_results.round(4))

## 7. Save `validation_results.csv` and comparison table

In [ ]:
validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

comparison_cols = [
    "rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison = validation_results[
    comparison_cols
].copy()

validation_comparison.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

display(validation_comparison.round(4))

## 8. Validation confusion matrices

In [ ]:
for _, row in validation_results.iterrows():
    key = row["result_key"]
    y_true, y_pred = prediction_store[key]

    labels = sorted(
        pd.Series(y_true).dropna().unique().tolist()
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"actual_{x}" for x in labels],
        columns=[f"predicted_{x}" for x in labels],
    )

    cm_df.to_csv(
        METRICS_DIR
        / f"{key}_validation_confusion_matrix.csv"
    )

    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels,
    ).plot(
        ax=ax,
        values_format="d",
    )

    ax.set_title(
        f"{row['dataset']} — {row['model']}"
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES_DIR
        / f"{key}_validation_confusion_matrix.png",
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

## 9. Validation comparison figure

In [ ]:
plot_df = validation_results.copy()

plot_df["candidate"] = (
    plot_df["dataset"].astype(str)
    + " | "
    + plot_df["model"].astype(str)
)

fig, ax = plt.subplots(
    figsize=(10, max(4, 0.5 * len(plot_df)))
)

ax.barh(
    plot_df["candidate"],
    plot_df["macro_f1"],
)

ax.set_xlabel("Validation Macro F1")
ax.set_ylabel("Dataset | Model")
ax.set_title("Validation Model Comparison")
ax.invert_yaxis()

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "validation_model_comparison.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

## 10. Select the best candidate model(s)

In [ ]:
best_by_dataset = (
    validation_results
    .sort_values(
        [
            "dataset",
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=[True, False, False],
    )
    .groupby(
        "dataset",
        as_index=False,
    )
    .first()
)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

display(
    best_by_dataset[
        [
            "dataset",
            "model",
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
        ]
    ].round(4)
)

best = validation_results.iloc[0]

print(
    f"Best overall candidate: "
    f"{best['model']} ({best['dataset']})"
)
print(
    f"Macro F1: {best['macro_f1']:.4f}"
)
print(
    f"Balanced accuracy: "
    f"{best['balanced_accuracy']:.4f}"
)

## 11. Validation metrics summary and justification

In [ ]:
display(
    validation_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

print("\nJUSTIFICATION")
print(
    f"{best['model']} on {best['dataset']} "
    f"is the strongest candidate to move forward "
    f"because it achieved the highest validation "
    f"Macro F1 ({best['macro_f1']:.4f}). "
    f"Its balanced accuracy was "
    f"{best['balanced_accuracy']:.4f}. "
    f"Macro F1 is used as the primary selection "
    f"metric because it gives equal importance to "
    f"performance across classes, while balanced "
    f"accuracy is used as the secondary comparison."
)

## 12. Deliverables check

In [ ]:
deliverables = pd.DataFrame([
    {
        "deliverable": "validation_results.csv",
        "path": METRICS_DIR / "validation_results.csv",
    },
    {
        "deliverable": "Validation comparison table",
        "path": TABLES_DIR / "validation_comparison.csv",
    },
    {
        "deliverable": "Best candidate table",
        "path": TABLES_DIR / "validation_best_models.csv",
    },
    {
        "deliverable": "Validation comparison figure",
        "path": FIGURES_DIR / "validation_model_comparison.png",
    },
])

deliverables["status"] = deliverables["path"].apply(
    lambda p: (
        "READY"
        if Path(p).exists()
        else "MISSING"
    )
)

display(deliverables)

print(
    "Confusion-matrix CSVs:",
    len(
        list(
            METRICS_DIR.glob(
                "*_validation_confusion_matrix.csv"
            )
        )
    ),
)

print(
    "Confusion-matrix figures:",
    len(
        list(
            FIGURES_DIR.glob(
                "*_validation_confusion_matrix.png"
            )
        )
    ),
)